# Simple RecSys Application First

In [1]:
import numpy as np, pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName ("Matrix Work") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

In [ ]:
#print(pair.count())
#print(bridge.count())
#print(bridge.select("pid", "pos").distinct().count())
#print(bridge.filter(F.col("track_uri").isNull() | (F.col("track_uri") == "")).count())   # must be 0

rep = (bridge.groupBy("pid", "track_uri")
       .agg(F.count("*").alias("n"), F.countDistinct("pos").alias("n_pos")).orderBy(F.col("n_pos").desc()))

rep.show()

+------+--------------------+---+-----+
|   pid|           track_uri|  n|n_pos|
+------+--------------------+---+-----+
| 67052|spotify:track:1mr...|138|  138|
|477863|spotify:track:2Wf...| 57|   57|
|218358|spotify:track:2AG...| 56|   56|
|480141|spotify:track:5KY...| 55|   55|
|331576|spotify:track:2rs...| 55|   55|
|652175|spotify:track:7Kc...| 54|   54|
|819513|spotify:track:5fp...| 51|   51|
|481135|spotify:track:4TV...| 50|   50|
|481135|spotify:track:59J...| 50|   50|
|481135|spotify:track:3mA...| 50|   50|
|682157|spotify:track:2HW...| 48|   48|
|437502|spotify:track:3fq...| 47|   47|
|626358|spotify:track:5Bk...| 39|   39|
|873919|spotify:track:3XH...| 33|   33|
|626358|spotify:track:5MI...| 33|   33|
|811998|spotify:track:14W...| 30|   30|
|626358|spotify:track:1Xy...| 30|   30|
|408204|spotify:track:3Ep...| 30|   30|
|461736|spotify:track:2LQ...| 30|   30|
|906273|spotify:track:4DM...| 29|   29|
+------+--------------------+---+-----+
only showing top 20 rows


In [11]:
rep = (bridge.groupBy("pid", "track_uri")
             .agg(F.count("*").alias("n"), F.countDistinct("pos").alias("n_pos"))
             .filter("n > 1"))
rep.agg(F.sum(F.col("n") - 1), F.count("*"), F.max("n"),
        F.sum(F.when(F.col("n_pos") != F.col("n"), 1).otherwise(0))).show()

+------------+--------+------+--------------------------------------------------+
|sum((n - 1))|count(1)|max(n)|sum(CASE WHEN (NOT (n_pos = n)) THEN 1 ELSE 0 END)|
+------------+--------+------+--------------------------------------------------+
|      881652|  829665|   138|                                                 0|
+------------+--------+------+--------------------------------------------------+



In [ ]:
# Collaborative Filtering Approach
# Lets start by generating a matrix of our data. Rows will represent tracks and columns will represent playlists
# For simplicity, we will be using 1s and 0s

# The size of this array is expected to be 2262292 x 1000000 - how to handle array of this size?
# Numpy uses RAM allocation for arrays...meaning trying to do this locally- you'd likely crash 2.26M x 1M  is about 2.26 trillion matrix cells...
# Bring in SciPy

# We can use our bridge table to get distinct pairs, that will help the matrix construction, we'll use the distinct function to be sure track and playlist pairs or unique

bridge = spark.read.parquet("../silver/pid_pos") 
pair = bridge.select("pid", "track_uri").distinct() #we want to turn this into a matrix

# we can use pid itself as the index for playlists 0-999999
# that leaves us with having to index tracks - which should match out track count -> 2262292

w = Window.orderBy("track_uri") #defining window spec

#the following will create a dictionary that assigns an index to the distinct tracks (counts should match our silver count)
track_index = (pair.select("track_uri").distinct()
                   .withColumn("track_idx", F.row_number().over(w) - 1))

print(track_index.count())
track_index.show() #checks out with our track count


2262292
+--------------------+---------+
|           track_uri|track_idx|
+--------------------+---------+
|spotify:track:000...|        0|
|spotify:track:000...|        1|
|spotify:track:000...|        2|
|spotify:track:000...|        3|
|spotify:track:000...|        4|
|spotify:track:000...|        5|
|spotify:track:000...|        6|
|spotify:track:000...|        7|
|spotify:track:000...|        8|
|spotify:track:000...|        9|
|spotify:track:000...|       10|
|spotify:track:000...|       11|
|spotify:track:000...|       12|
|spotify:track:000...|       13|
|spotify:track:000...|       14|
|spotify:track:000...|       15|
|spotify:track:000...|       16|
|spotify:track:000...|       17|
|spotify:track:000...|       18|
|spotify:track:000...|       19|
+--------------------+---------+
only showing top 20 rows


In [ ]:
#now we will start adding to our gold layer

#interactions will serve to map playlist id to the indices we created with track_index
interactions = (pair.join(track_index, on='track_uri', how='inner')
                .select(F.col('pid').cast('int'), F.col('track_idx'))) #only selecting the indices that will be fed to sparse matrix (pid and track_idx)

interactions.show() #showing indices tracks mapped to pid, to show unique track and playlist interactions... will also be what twe use to make the sparse matrix

+------+---------+
|   pid|track_idx|
+------+---------+
|644281|        9|
|517405|        9|
|927842|       27|
|515672|       27|
|629803|       27|
| 80001|       27|
|720299|       48|
| 98329|       91|
|225981|       95|
|633379|      144|
|870868|      150|
|555959|      157|
|217393|      165|
|475201|      165|
|662403|      206|
|528896|      206|
|245117|      313|
|872490|      443|
|998975|      478|
|921677|      486|
+------+---------+
only showing top 20 rows


In [ ]:
#Lets persist
track_index.coalesce(1).write.mode("overwrite").parquet("../gold/track_index") #write reasoning for partition size
interactions.coalesce(8).write.mode("overwrite").parquet("../gold/interactions") 

In [ ]:
#convertying to numpy... explain why
df = pd.read_parquet("../gold/interactions")

rows = df["track_idx"].to_numpy()
cols = df["pid"].to_numpy()

print(rows)
print(cols)

[      9       9      27 ... 2262252 2262255 2262279]
[644281 517405 927842 ... 101874 834940 650250]


In [ ]:
#constructing the matrix according to csr_array documentation (compressed sparse row array...went with row since it contains more values than column)

#explain csr choice vs csc
#using the csr_array class, the csr_array object class has the following parameters
# csr_array((data, indices)[shape = (M,N)]), all of which are 2 synchronized 1D arrays 

from scipy.sparse import csr_array

N_TRACKS = 2_262_292
N_PLAYLISTs = 1_000_000

M = csr_array(
    (np.ones(len(df), dtype=np.float32), #data for coordinates ... float 32 to account for matrix multiplication in float
    (rows,cols)), #coordinate pairs that we got from the interactions table we created earlier 
    shape=(N_TRACKS, N_PLAYLISTs)
)

print(M)

<Compressed Sparse Row sparse array of dtype 'float32'
	with 65464776 stored elements and shape (2262292, 1000000)>
  Coords	Values
  (0, 153150)	1.0
  (1, 262470)	1.0
  (1, 534798)	1.0
  (1, 775769)	1.0
  (1, 805462)	1.0
  (1, 887491)	1.0
  (1, 915858)	1.0
  (2, 127649)	1.0
  (2, 592055)	1.0
  (3, 271291)	1.0
  (4, 832997)	1.0
  (5, 617677)	1.0
  (6, 184863)	1.0
  (7, 75952)	1.0
  (7, 216958)	1.0
  (7, 247450)	1.0
  (7, 254696)	1.0
  (7, 274839)	1.0
  (7, 287991)	1.0
  (7, 294496)	1.0
  (7, 312257)	1.0
  (7, 569193)	1.0
  (7, 580913)	1.0
  (7, 581520)	1.0
  (7, 634104)	1.0
  :	:
  (2262281, 720033)	1.0
  (2262281, 879861)	1.0
  (2262281, 891180)	1.0
  (2262281, 910228)	1.0
  (2262281, 952274)	1.0
  (2262281, 992447)	1.0
  (2262281, 994517)	1.0
  (2262282, 37124)	1.0
  (2262283, 641318)	1.0
  (2262283, 706566)	1.0
  (2262284, 676411)	1.0
  (2262285, 273825)	1.0
  (2262285, 560381)	1.0
  (2262286, 131838)	1.0
  (2262287, 538214)	1.0
  (2262287, 547474)	1.0
  (2262287, 968984)	1.0
  (226

In [12]:
print(M[:, [4242]].sum())
print(pair.filter('pid=4242').count())     # list index keeps it 2-D and version-stable

39.0
39


In [ ]:
#we cannot compute the matrix everytime, as an example, we will showcase a popular track and it's co occurences from the matrix we just created 

df = spark.read.parquet("../silver/pid_pos")

pop = df.groupBy("track_uri").agg(F.count("track_uri").alias("count")).orderBy(F.col("count").desc()) #we are querying the pid_pos, which give us ALL playlist and track combinations
#we then group by track_uris and aggregate on the track_uri counts to see which track appeared the most across the million playlists 
pop.show(truncate=False)

+------------------------------------+-----+
|track_uri                           |count|
+------------------------------------+-----+
|spotify:track:7KXjTSCq5nL1LoYtL7XAwS|46574|
|spotify:track:1xznGGDReH1oQq0xzbwXa3|43447|
|spotify:track:7yyRTcZmCiyzzJlNzGC9Ol|41309|
|spotify:track:7BKLCZ1jbUBVqRi2FVlTVw|41079|
|spotify:track:3a1lNhkSLSkpJE4MSHpDu9|39987|
|spotify:track:5hTpBe8h35rJ67eAWHQsJx|35202|
|spotify:track:2EEeOnHehOozLq4aS0n6SL|35138|
|spotify:track:4Km5HrUvYTaSUfiSGPJeQR|34999|
|spotify:track:7GX5flRQZVHRAGd6B4TmDO|34922|
|spotify:track:152lZdxL1OR0ZMW6KquMif|34657|
|spotify:track:0SGkqnVQo9KPytSri1H6cF|33699|
|spotify:track:5dNfHmqgr128gMY2tc5CeJ|32391|
|spotify:track:62vpWI1CHwFy7tMIcSStl8|32336|
|spotify:track:0VgkVdmE4gld66l8iyGjgx|32059|
|spotify:track:0v9Wz8o0BT8DU38R4ddjeH|31492|
|spotify:track:3DXncPQOG4VBw3QHh3S817|31374|
|spotify:track:27GmP9AWRs744SzKcpJsTZ|31119|
|spotify:track:6gBFPUFcJLzWGx4lenP6h2|31106|
|spotify:track:343YBumqHu19cGoGARUTsd|30678|
|spotify:t

In [ ]:
track_index.filter("track_uri='spotify:track:7KXjTSCq5nL1LoYtL7XAwS'").show() # getting track index for popular track

+--------------------+---------+
|           track_uri|track_idx|
+--------------------+---------+
|spotify:track:7KX...|  2128659|
+--------------------+---------+



In [ ]:
#We cannot calculate the entire co-occurence matrix, hence, we will only care about the row that represent the track
#we're interested in finding co-occurrences about
i = 2128659
playlists_with_i = M[[i]] #row vector [1 0 1 1 1 0 ... 0]

#need to document linear algebra here
c = (M @ playlists_with_i.T).toarray().ravel() # gives us tracks that appear together with the specified i= 2128659 track 
deg = M.sum(axis=1) #axis = 1 specifies rows to be summed - though they should already be summed since the row vector transpose is of the form 1000000x1 

c[i]=0 #assigning 0 to itself

top = np.argsort(c)[::-1][:10] #review syntax -> Array is sorted ascending, [::-1] reverses it 
#took 0.7 seconds of computation

top_to_uri = track_index.filter(F.col("track_idx").isin(top))

tracks = pd.read_parquet("../silver/track", columns=["track_uri", "track_name"])
out = pd.DataFrame({"track_uri":top_to_uri, "shared_playlists": c[top]})

out.merge(tracks, on="track_uri")

{"ts": "2026-09-05 10:57:25.438", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.DATA_DIFF_TYPES] Cannot resolve \"(track_idx IN (ARRAY(1825557L, 2109995L, 149078L, 1041889L, 133008L, 1940453L, 649447L, 1261050L, 1981992L, 1286037L)))\" due to data type mismatch: Input to `in` should all be the same type, but it's [\"INT\", \"ARRAY<BIGINT>\"]. SQLSTATE: 42K09", "context": {"file": "line 14 in cell [23]", "line": "", "fragment": "isin", "errorClass": "DATATYPE_MISMATCH.DATA_DIFF_TYPES"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o205.filter.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.DATA_DIFF_TYPES] Cannot resolve \"(track_idx IN (ARRAY(1825557L, 2109995L, 149078L, 1041889L, 133008L, 1940453L, 649447L, 1261050L, 1981992L, 1286037L)))\" due to data type mismatch: Input to `in` should all be the same type, but it's [\"INT\", \"ARRAY<BIGINT>\"]. SQLSTATE: 42K09;\n'Filter track_idx#46 IN ([18255

AnalysisException: [DATATYPE_MISMATCH.DATA_DIFF_TYPES] Cannot resolve "(track_idx IN (ARRAY(1825557L, 2109995L, 149078L, 1041889L, 133008L, 1940453L, 649447L, 1261050L, 1981992L, 1286037L)))" due to data type mismatch: Input to `in` should all be the same type, but it's ["INT", "ARRAY<BIGINT>"]. SQLSTATE: 42K09;
'Filter track_idx#46 IN ([1825557,2109995,149078,1041889,133008,1940453,649447,1261050,1981992,1286037])
+- Project [track_uri#44, track_idx#46]
   +- Project [track_uri#44, _we0#48, (_we0#48 - 1) AS track_idx#46]
      +- Window [row_number() windowspecdefinition(track_uri#44 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS _we0#48], [track_uri#44 ASC NULLS FIRST]
         +- Project [track_uri#44]
            +- Deduplicate [track_uri#44]
               +- Project [track_uri#44]
                  +- Deduplicate [pid#42L, track_uri#44]
                     +- Project [pid#42L, track_uri#44]
                        +- Relation [pid#42L,p

In [ ]:
ti = pd.read_parquet("../gold/track_index").sort_values("track_idx")
idx_to_uri = ti["track_uri"].to_numpy()          # position == track_idx

tracks = pd.read_parquet("../silver/track", columns=["track_uri", "track_name"])
out = pd.DataFrame({"track_uri": idx_to_uri[top], "shared_playlists": c[top]})
out.merge(tracks, on="track_uri")

#need to derive this on own and understand mistakes from top code error
#this shows us the top tracks that co occured with the track we first looked up

,track_uri,shared_playlists,track_name
0,spotify:track:6HZILIRieu8S0iqY8kIKhj,21146.0,DNA.
1,spotify:track:7GX5flRQZVHRAGd6B4TmDO,19953.0,XO TOUR Llif3
2,spotify:track:0VgkVdmE4gld66l8iyGjgx,19732.0,Mask Off
3,spotify:track:3a1lNhkSLSkpJE4MSHpDu9,19592.0,Congratulations
4,spotify:track:0SGkqnVQo9KPytSri1H6cF,15948.0,Bounce Back
5,spotify:track:6gBFPUFcJLzWGx4lenP6h2,15913.0,goosebumps
6,spotify:track:2EEeOnHehOozLq4aS0n6SL,15700.0,iSpy (feat. Lil Yachty)
7,spotify:track:4Km5HrUvYTaSUfiSGPJeQR,15275.0,Bad and Boujee (feat. Lil Uzi Vert)
8,spotify:track:6p8NuHm8uCGnn2Dtbtf7zE,14463.0,Slippery (feat. Gucci Mane)
9,spotify:track:4Q3N4Ct4zCuIHuZ65E3BD4,13706.0,Tunnel Vision
